# Cancellation-rate sensitivity

Hold initial quantity, arrival rate, consumption rate and cutoff fixed. This finite parameter sweep illustrates an effect; it is not a proof of general monotonicity.

Synthetic data only. All settings and random seeds are specified below.

In [1]:
from pathlib import Path
import sys, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "results" / "synthetic" / "queues"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": .2})
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})


{'python': '3.12.0', 'numpy': '1.26.0', 'pandas': '2.1.1'}


In [2]:
from queue_models.replenishment import Config,simulate,finite_reference
from queue_models.queue_depletion import QueueConfig,simulate as zero_sim,reference,summarize
rows=[]
for theta in (0.,.02,.1,.2,.5,1.):
    if theta==0:
        c=QueueConfig(3,1.,1.);sample=zero_sim(c,horizon=100.,samples=5000,seed=510);ref=reference(c,[100.])[0]
    else:
        c=Config(3,1.,1.,theta);sample=simulate(c,[100.],samples=5000,seed=510)[0];ref=finite_reference(c,[100.])[0]
    rows.append(dict(theta=theta,**summarize(sample,100.),reference_restricted_mean=ref["restricted_mean"]))
table=pd.DataFrame(rows);display(table)
table.to_csv(OUT/"theta_sweep.csv",index=False)
plt.plot(table.theta,table.restricted_mean,"o-",label="Simulation")
plt.plot(table.theta,table.reference_restricted_mean,"k--",label="Reference")
plt.xlabel("Per-order cancellation rate");plt.ylabel("E[min(T,100)]");plt.legend()
plt.tight_layout();plt.savefig(OUT/"theta_sweep.png",dpi=140);plt.show()


,theta,cutoff,samples,depleted,censored,survival,survival_low,survival_high,restricted_mean,restricted_se,completed_only_mean,reference_restricted_mean
0,0.00,100.0,5000,4136,864,0.1728,0.152465,0.195222,30.159774,0.520353,15.570327,29.597792
1,0.02,100.0,5000,4775,225,0.0450,0.034654,0.058249,19.883895,0.373903,16.108791,20.044314
2,0.10,100.0,5000,5000,0,0.0000,0.000000,0.003190,8.317794,0.130018,8.317794,8.197685
3,0.20,100.0,5000,5000,0,0.0000,0.000000,0.003190,5.066641,0.065634,5.066641,5.092021
4,0.50,100.0,5000,5000,0,0.0000,0.000000,0.003190,2.592427,0.029770,2.592427,2.569904
5,1.00,100.0,5000,5000,0,0.0000,0.000000,0.003190,1.459070,0.014756,1.459070,1.464536


/var/folders/p2/_bpfxjzn1l301l3jgtn6bgx80000gn/T/ipykernel_57339/1989989774.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.savefig(OUT/"theta_sweep.png",dpi=140);plt.show()


## Interpretation

Compare the displayed model reference with the simulation. Finite-sample agreement is an implementation check, not evidence of real-market calibration. See [assumptions](../../../docs/model_assumptions.md) and [limitations](../../../docs/validation_and_limitations.md).